In [1]:
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)


import os
from langchain_community.document_loaders import (
    PyPDFLoader,
    PyMuPDFLoader
)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

c:\Users\abdul\OneDrive\Desktop\Agentic\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


`Get the PDF Path -> List of all pdfs -> for each pdf -> documentize -> store in all_documents`

In [2]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory: str) -> list:

    all_documents = []  # Will save all the documents
    pdf_dir = Path(pdf_directory)

    # Find all pdfs recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf")) # [andrew_ng.pdf, bigdata.pdf] path objects

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"Processing {pdf_file.name}")

        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load() # list of all documents (each page of a pdf is a document)

            # add sourcre data to meta data
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type']= 'pdf'

            all_documents.extend(documents)
            print(f"✔️ Loaded {len(documents)} pages")

        except Exception as e:
            print(f"❌ Error: {e}")

    print(f"Total documents loaded: {len(all_documents)}")
    return all_documents

In [3]:
docs = process_all_pdfs("../data/pdfs")

Found 2 PDF files to process
Processing andrewNG.pdf
✔️ Loaded 118 pages
Processing ML with Big Data.pdf
✔️ Loaded 22 pages
Total documents loaded: 140


`Takes list of all documents -> Splits them into chunks (list of smaller/broken down documents)`

In [4]:
## Chunking

def chunk_documents(documents, chunk_size = 1000, chunk_overlap = 200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators = ["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    return split_docs

In [5]:
chunks = chunk_documents(docs, chunk_size=1000, chunk_overlap=200)

Split 140 documents into 396 chunks


In [6]:
chunks[20]

Document(metadata={'producer': 'Adobe PDF library 16.03', 'creator': 'Adobe Illustrator 26.0 (Windows)', 'creationdate': '2022-03-02T11:13:23+06:00', 'moddate': '2022-03-02T11:13:23+05:00', 'title': 'andrew-ng-machine-learning-yearning', 'source': '..\\data\\pdfs\\andrewNG.pdf', 'total_pages': 118, 'page': 14, 'page_label': '15', 'source_file': 'andrewNG.pdf', 'file_type': 'pdf'}, page_content='In other words, \u200bthe purpose of the dev and test sets are to direct your team toward \nthe most important changes to make to the machine learning system\u200b.  \nSo, you should do the following:  \nChoose dev and test sets to reflect data you expect to get in the future \nand want to do well on.  \nIn other words, your test set should not simply be 30% of the available data, especially if you \nexpect your future data (mobile phone images) to be different in nature from your training \nset (website images).  \nIf you have not yet launched your mobile app, you might not have any users yet, 

### Embedding manager

In [7]:
import numpy as np
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

`Automatically loads the model when instantiated. generate_embeddings: list[smaller documents] -> embedding mapping`

In [8]:
class EmbeddingModel:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model() 

    def _load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}\n")
        except Exception as e:
            print(f"Error loading model {self.model_name} : {e}")
            raise e

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model is not loaded. Please call _load_model() first")

        print(f"Generating embeddings for {len(texts)} texts...\n")
        embeddings = self.model.encode(texts, show_progress_bar=True, convert_to_numpy=True)
        print(f"Generated embeddings with shape: {embeddings.shape}\n")
        return embeddings

In [9]:
model = EmbeddingModel(model_name="all-MiniLM-L6-v2")

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10292.16it/s]


Model loaded successfully. Embedding dimension: 384



### Vector Store

`Defines the collection_name, persistent directory and the client when an object is made. add_embeddings takes the list embedding vectors and list of chunks and stores them in the db along with their respective documents`

In [10]:
class VectorStore:

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../vectorstore"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_chromadb()

    def _initialize_chromadb(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path = self.persist_directory)

            self.collection = self.client.get_or_create_collection( # gets the existing collection if exists
                name = self.collection_name,
                metadata = {"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized with collection: {self.collection_name}\n")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing ChromaDB vector store : {e}")
            raise e

    def add_embeddings(self, documents: List[Any], embeddings: np.ndarray):
        """
            Add docs and their embeddings to the store. Each document should have a unique ID, content, and metadata.
        """

        if len(documents) != embeddings.shape[0]:
            raise ValueError("Number of documents and embeddings must match")

        # Edge case: collection already contains embeddings
        if self.collection.count() > 0:
            print("Embeddings already exist in the vector store. Skipping insertion.")
            print("Total documents in collection:", self.collection.count())
            return

        print(f"Adding {len(documents)} documents to the vector store...\n")

        # prepare data for insertion
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):

            # generate a unique ID for each document
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare the metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Add the document text
            documents_text.append(doc.page_content)

            # Add the embedding
            embeddings_list.append(embedding.tolist())

        # Add to the collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to the vector store.\n")
            print("Total documents in collection after addition: ", self.collection.count())

        except Exception as e:
            print(f"Error adding documents to the vector store : {e}")
            raise e

In [11]:
model = EmbeddingModel(model_name="all-MiniLM-L6-v2")
embeddings = model.generate_embeddings([chunk.page_content for chunk in chunks])

vectorstore = VectorStore(collection_name="pdf_documents", persist_directory="../vectorstore")
vectorstore.add_embeddings(chunks, embeddings)

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10275.27it/s]


Model loaded successfully. Embedding dimension: 384

Generating embeddings for 396 texts...



Batches: 100%|██████████| 13/13 [00:03<00:00,  3.49it/s]


Generated embeddings with shape: (396, 384)

Vector store initialized with collection: pdf_documents

Existing documents in collection: 396
Embeddings already exist in the vector store. Skipping insertion.
Total documents in collection: 396


### Retriever

In [19]:
class RAGRetriever:
    """
        Handles query-based retrieval from the vector store
    """

    def __init__(self, vector_store: VectorStore, embedding_model: EmbeddingModel):
        self.vector_store = vector_store
        self.embedding_model = embedding_model

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
            Retrieves relevant documents from a query 

        """
        
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K : {top_k}, Score Threshold: {score_threshold}")

        # embed the query
        query_embedding = self.embedding_model.generate_embeddings([query])[0]

        # Search in vector db
        try:
            results = self.vector_store.collection.query(
                query_embeddings = [query_embedding.tolist()],
                n_results = top_k
            )

            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                # if results and documents are not empty
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_text, metadata, distance, doc_id) in enumerate(zip(documents, metadatas, distances, ids)):
                    similarity_score = 1 - distance  # Convert distance to similarity score
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": doc_text,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })
            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            raise e

from rank_bm25 import BM25Okapi
import re

class BM25Retriever:
    def __init__(self, documents: List[Any]):
        """documents = your list of chunked LangChain Document objects"""
        self.documents = documents
        # Tokenize each document's content
        self.tokenized_corpus = [self._tokenize(doc.page_content) for doc in documents]
        self.bm25 = BM25Okapi(self.tokenized_corpus)

    def _tokenize(self, text: str) -> list:
        """Simple whitespace + lowercase tokenizer"""
        return re.sub(r'[^\w\s]', '', text.lower()).split()

    def retrieve(self, query: str, top_k: int = 5) -> list:
        tokenized_query = self._tokenize(query)
        scores = self.bm25.get_scores(tokenized_query)

        # Get top-k indices
        top_indices = scores.argsort()[::-1][:top_k]

        results = []
        for rank, idx in enumerate(top_indices):
            results.append({
                "id": f"bm25_{idx}",
                "content": self.documents[idx].page_content,
                "metadata": dict(self.documents[idx].metadata),
                "bm25_score": float(scores[idx]),
                "rank": rank + 1
            })
        return results


In [20]:
class HybridRetriever:
    def __init__(self, vector_retriever: RAGRetriever, bm25_retriever: BM25Retriever):
        self.vector_retriever = vector_retriever
        self.bm25_retriever = bm25_retriever

    def retrieve(self, query: str, top_k: int = 5, k: int = 60) -> list:
        """Reciprocal Rank Fusion (RRF) to combine results"""
        vector_results = self.vector_retriever.retrieve(query, top_k=top_k * 2)
        bm25_results = self.bm25_retriever.retrieve(query, top_k=top_k * 2)

        # Build RRF score map: score = 1 / (k + rank)
        rrf_scores = {}  # content -> {score, doc_info}

        for result in vector_results:
            key = result["content"]
            rrf_scores[key] = rrf_scores.get(key, {"score": 0, "doc": result})
            rrf_scores[key]["score"] += 1.0 / (k + result["rank"])

        for result in bm25_results:
            key = result["content"]
            rrf_scores[key] = rrf_scores.get(key, {"score": 0, "doc": result})
            rrf_scores[key]["score"] += 1.0 / (k + result["rank"])

        # Sort by fused score and return top_k
        sorted_results = sorted(rrf_scores.values(), key=lambda x: x["score"], reverse=True)[:top_k]

        return [
            {**item["doc"], "rrf_score": item["score"], "rank": i + 1}
            for i, item in enumerate(sorted_results)
        ]


In [ ]:

bm25_retriever = BM25Retriever(chunks)
vector_retriever = RAGRetriever(vector_store=vectorstore, embedding_model=model)
hybrid_retriever = HybridRetriever(vector_retriever, bm25_retriever)


Retrieving documents for query: 'What is overfitting?'
Top K : 10, Score Threshold: 0.0
Generating embeddings for 1 texts...



Batches: 100%|██████████| 1/1 [00:00<00:00, 60.92it/s]

Generated embeddings with shape: (1, 384)



In [28]:
results = hybrid_retriever.retrieve("What is the curse of dimensionality?")
for result in results:
    print("=" * 30 , result['metadata']['source_file'], "=" * 30)
    print(result['content'] + "\n")


Retrieving documents for query: 'What is the curse of dimensionality?'
Top K : 10, Score Threshold: 0.0
Generating embeddings for 1 texts...



Batches: 100%|██████████| 1/1 [00:00<00:00, 58.18it/s]

Generated embeddings with shape: (1, 384)

============================== ML with Big Data.pdf ==============================
A. L’Heureux et al.: Machine Learning With Big Data: Challenges and Approaches
gamiﬁcation and demonstrated its negative effects on Watson
machine learning.
Consequently, in the Big Data context, due to data size, the
probability that class imbalance will occur is high. In addi-
tion, because of the complex problems embedded in such
data, the potential effects of class imbalance on machine
learning are severe.
4) CURSE OF DIMENSIONALITY
Another issue associated with the volume of Big Data is
the curse of dimensionality [37] which refers to difﬁcul-
ties encountered when working in high dimensional space.
Speciﬁcally, the dimensionality describes the number of
features or attributes present in the dataset. The Hughes
effect [38] states that for a training set of static size, the pre-
dictive ability and effectiveness of an algorithm decreases
as the dimensionalit